#Init


In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import StringType
from pyspark.sql.functions import col, trim

In [0]:
RENAME_MAP = {
    "cst_id": "customer_id",
    "cst_key": "customer_key",
    "cst_firstname": "first_name",
    "cst_lastname": "last_name",
    "cst_marital_status": "marital_status",
    "cst_gndr": "gender",
    "cst_create_date": "create_date"
}

#Reading from bronze layer

In [0]:
df = spark.read.table("workspace.bronze.crm_cust_info")

#Data transformations

data quality issues
- Trim the string columns
- Column names
- normalization marital status, gender
- table name not friendly
- null values
- 

##Trim string columns

In [0]:

for field in df.schema.fields:
    if isinstance(field.dataType, StringType):
        df = df.withColumn(field.name, trim(col(field.name))) 



##Normalization of few columns

In [0]:


df = (
    df
    .withColumn("cst_marital_status", 
    F.when(col("cst_marital_status").isin("M", "Married"), "Married")
     .when(col("cst_marital_status").isin("S", "Single"), "Single")
     .otherwise("Unknown")
    )
    .withColumn("cst_gndr", 
    F.when(col("cst_gndr").isin("M", "Male"), "Male")
     .when(col("cst_gndr").isin("F", "Female"), "Female")
     .otherwise("Unknown"))
)



##Renaming the columns

In [0]:
for old_name, new_name in RENAME_MAP.items():
    df = df.withColumnRenamed(old_name, new_name)

#Writing into silver table

In [0]:
(df.write
    .mode("overwrite")
    .format("delta")
    .saveAsTable("silver.crm_customers")
)